# 09 — Report assets

Reads every `metrics.json` under `/content/results` (smoke runs excluded) and produces, in `/content/results/report/`:

1. `results_table.csv` / `results_table.md` — every run, all metrics.
2. `ball_map_bar.png` — ball mAP50-95 per experiment, the headline figure. Grey band = run-to-run noise (|02 − 02b|) around the baseline.
3. `ball_map_vs_fps.png` — accuracy against inference speed: each intervention's cost in one picture.
4. `failure_gallery.png` — test-split balls the baseline misses (no ball prediction with IoU ≥ 0.5 at conf 0.25), native-resolution crops, one-line cause each. Causes are written after looking at the crops (`CAUSES`).

Colours: reference palette slot 1 (single series, direct labels, no legend needed).

In [ ]:
RESULTS = "/content/results"
OUT = "/content/results/report"
BASELINE = "baseline_yolo11s_640"
SEED_TWIN = "baseline_yolo11s_640_seed1"
# gallery index -> one-line cause, written after inspecting the crops (2026-09-26)
CAUSES = {
    0: 'isolated in open grass, clearly visible',
    1: "at a player's feet, on the touchline",
    2: "at a player's feet",
    3: "occluded by a player's leg",
    4: 'on a painted line near the byline',
    5: 'in front of the goal, on the goal line',
    6: "among several players' feet",
    7: 'motion-blurred streak in sunlight',
    8: 'beside two players, sun/shade edge',
    9: 'on a painted line in bright sun',
    10: 'in dark shadow, low contrast',
    11: 'isolated in shade, clearly visible',
    12: 'isolated in open grass, clearly visible',
    13: "at a player's feet, partly occluded",
    14: 'far touchline, a few pixels wide',
    15: 'motion blur, far side',
}

In [ ]:
import sys, json, pathlib
sys.path.insert(0, "/content/ballhawk")
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import ballhawk_common as bc

SERIES, REF, INK, INK2, GRID, SURFACE, BAND = "#2a78d6", "#8f8e89", "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb", "#d9d8d3"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": GRID,
                     "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
                     "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
out = pathlib.Path(OUT); out.mkdir(parents=True, exist_ok=True)

rows = []
for f in sorted(pathlib.Path(RESULTS).glob("*/metrics.json")):
    r = json.loads(f.read_text())
    if r["run_name"].endswith("_smoke"):
        continue
    ball = next(c for c in r["per_class"] if c["class"] == "ball")
    rows.append({"notebook": r["notebook"], "run_name": r["run_name"], "variable_changed": r["variable_changed"],
                 "imgsz": r["imgsz"], "batch": r["batch"], "mAP50": r["overall"]["mAP50"],
                 "mAP50_95": r["overall"]["mAP50_95"], "ball_P": ball["precision"], "ball_R": ball["recall"],
                 "ball_mAP50": ball["mAP50"], "ball_mAP50_95": ball["mAP50_95"],
                 "fps": r["inference_fps"], "train_min": r["train_minutes"], "notes": r["notes"]})
df = pd.DataFrame(rows).sort_values(["notebook", "run_name"]).reset_index(drop=True)
SHORT = {"baseline_yolo11s_640": "02 baseline", "baseline_yolo11s_640_seed1": "02b baseline, seed 1",
         "res_yolo11s_960": "03 imgsz 960", "res_yolo11s_1280": "03 imgsz 1280 (batch 8)",
         "aug_scale02_yolo11s_640": "04a scale 0.2", "aug_scale02_mblur_yolo11s_640": "04b 04a + motion blur",
         "p2_yolo11s_640": "05 P2 head", "clsw05_yolo11s_640": "05b class weights",
         "ctrl_reinit17_yolo11s_640": "05c control: head re-init", "sahi_baseline_s960": "06 SAHI 960 on 02"}
REFERENCE = {"baseline_yolo11s_640", "baseline_yolo11s_640_seed1", "ctrl_reinit17_yolo11s_640"}
df["label"] = [SHORT.get(r, f"{n} {v[:28]}") for r, n, v in zip(df.run_name, df.notebook, df.variable_changed)]
df["colour"] = [REF if r in REFERENCE else SERIES for r in df.run_name]
df.to_csv(out / "results_table.csv", index=False)
num = df.drop(columns=["notes"]).round(3)
md_lines = ["| " + " | ".join(num.columns) + " |", "|" + "---|" * len(num.columns)]
md_lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in num.itertuples(index=False)]
(out / "results_table.md").write_text("\n".join(md_lines) + "\n")
num

In [ ]:
from matplotlib.patches import Patch
base = df.set_index("run_name").loc[BASELINE, "ball_mAP50_95"]
noise = abs(base - df.set_index("run_name").loc[SEED_TWIN, "ball_mAP50_95"]) if SEED_TWIN in set(df.run_name) else None

fig, ax = plt.subplots(figsize=(8, 0.42 * len(df) + 1.6))
y = np.arange(len(df))[::-1]
if noise is not None:
    ax.axvspan(base - noise, base + noise, color=BAND, alpha=0.7, lw=0)
ax.axvline(base, color=INK2, lw=1, ls="--")
ax.barh(y, df.ball_mAP50_95, height=0.62, color=df.colour)
for yi, v in zip(y, df.ball_mAP50_95):
    ax.text(v + 0.004, yi, f"{v:.3f}", va="center", color=INK, fontsize=9)
ax.set_yticks(y, df.label, fontsize=9)
ax.set_xlabel("ball mAP50-95 (test split)")
ax.set_xlim(0, df.ball_mAP50_95.max() * 1.15)
ax.set_title("Ball detection by intervention", loc="left", color=INK, fontsize=12, pad=28)
ax.grid(axis="x", color=GRID, lw=0.8); ax.set_axisbelow(True)
handles = [Patch(color=SERIES, label="intervention"), Patch(color=REF, label="baseline / control")]
if noise is not None:
    handles.append(Patch(color=BAND, label=f"baseline ± seed noise ({noise:.3f})"))
ax.legend(handles=handles, loc="lower left", bbox_to_anchor=(0, 1.0), ncol=3, frameon=False, fontsize=8.5)
fig.tight_layout(); fig.savefig(out / "ball_map_bar.png", dpi=200)

In [ ]:
from matplotlib.ticker import FixedLocator, NullLocator, ScalarFormatter
OFFSET = {"aug_scale02_yolo11s_640": (8, 7), "aug_scale02_mblur_yolo11s_640": (-8, 9),
          "clsw05_yolo11s_640": (-8, -13), "baseline_yolo11s_640_seed1": (8, 3), "baseline_yolo11s_640": (8, -12),
          "p2_yolo11s_640": (-8, -4), "ctrl_reinit17_yolo11s_640": (8, 0)}
fig, ax = plt.subplots(figsize=(8, 5.2))
ax.scatter(df.fps, df.ball_mAP50_95, s=64, color=df.colour, edgecolor=SURFACE, linewidth=2, zorder=3)
for _, r in df.iterrows():
    dx, dy = OFFSET.get(r.run_name, (8, 4))
    ax.annotate(r.label, (r.fps, r.ball_mAP50_95), xytext=(dx, dy), textcoords="offset points", fontsize=8.5,
                color=INK, ha="left" if dx >= 0 else "right", va="center")
ax.set_xscale("log")
ax.xaxis.set_major_locator(FixedLocator([2, 5, 10, 20, 50]))
ax.xaxis.set_major_formatter(ScalarFormatter()); ax.xaxis.set_minor_locator(NullLocator())
ax.set_xlim(df.fps.min() * 0.7, df.fps.max() * 1.6)
ax.set_xlabel("inference FPS, batch 1, T4, idle VM, median of 3 (log scale)")
ax.set_ylabel("ball mAP50-95 (test split)")
ax.set_title("Accuracy against speed", loc="left", color=INK, fontsize=12)
ax.grid(color=GRID, lw=0.8); ax.set_axisbelow(True)
ax.legend(handles=[Patch(color=SERIES, label="intervention"), Patch(color=REF, label="baseline / control")],
          loc="upper left", frameon=False, fontsize=8.5)
fig.tight_layout(); fig.savefig(out / "ball_map_vs_fps.png", dpi=200)

In [ ]:
# Failure gallery: test balls the baseline misses at conf 0.25 (no ball box with IoU >= 0.5).
import torch
from PIL import Image
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou

data_yaml = bc.prepare_data()
model = YOLO(f"{RESULTS}/{BASELINE}/best.pt")
misses = []
for img in bc.split_images(data_yaml, "test"):
    w, h = Image.open(img).size
    lbl = img.parents[1] / "labels" / f"{img.stem}.txt"
    gt = [list(map(float, l.split()[1:])) for l in lbl.read_text().splitlines() if l.strip() and l.split()[0] == "0"]
    if not gt:
        continue
    gt = np.array([[(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h] for cx, cy, bw, bh in gt])
    b = model.predict(img, imgsz=640, conf=0.25, verbose=False)[0].boxes
    pred = b.xyxy[b.cls == 0].cpu()
    hit = (box_iou(torch.from_numpy(gt).float(), pred).max(1).values >= 0.5).numpy() if len(pred) else np.zeros(len(gt), bool)
    misses += [(img, box) for box, ok in zip(gt, hit) if not ok]
print(f"missed {len(misses)} test balls")

CROP, N = 96, 16
shown = misses[:N]
cols = 4; rows_ = int(np.ceil(len(shown) / cols))
fig, axes = plt.subplots(rows_, cols, figsize=(3 * cols, 3.3 * rows_))
for i, ax in enumerate(np.atleast_1d(axes).ravel()):
    ax.axis("off")
    if i >= len(shown):
        continue
    img, (x1, y1, x2, y2) = shown[i]
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    im = Image.open(img)
    left = int(np.clip(cx - CROP / 2, 0, im.width - CROP)); top = int(np.clip(cy - CROP / 2, 0, im.height - CROP))
    ax.imshow(im.crop((left, top, left + CROP, top + CROP)), interpolation="nearest")
    ax.add_patch(plt.Rectangle((x1 - left - 2, y1 - top - 2), x2 - x1 + 4, y2 - y1 + 4, fill=False, ec="#e34948", lw=1.2))
    ax.set_title(f"#{i} {CAUSES.get(i, img.stem.split('.rf.')[0])}", fontsize=8, color=INK)
fig.suptitle("Test-split balls missed by the baseline (96 px native crops, red = ground truth)", x=0.01, ha="left", fontsize=11)
fig.tight_layout(); fig.savefig(out / "failure_gallery.png", dpi=200)
pd.DataFrame([{"index": i, "image": m[0].name} for i, m in enumerate(shown)]).to_csv(out / "failure_gallery_index.csv", index=False)